# 03 - ControlNet Training (SD 1.5) - v4

Tuned for **RTX 5060 Laptop 8GB (Blackwell sm_120) / i7-14700HX / 16GB RAM**.

**Why your v3 ran at ~2 min/step, in order of blame:**

1. Full-resolution PNGs decoded on the **main thread** (`dataloader_num_workers` defaults to 0). Notebook 02 v3 now pre-resizes to 512 JPEG, and this notebook sets workers.
2. **No 8-bit Adam.** Regular AdamW holds ~2.9 GB of moment state for ControlNet's ~361M params. On 8 GB that pushed you into Windows' silent *system-RAM fallback* — the GPU shows 100% utilisation while it waits on PCIe. That is the 50x slowdown.
3. **Validation every 500 steps** builds a whole pipeline in VRAM each time and can leave you spilled afterwards.

**Also fixed:** dataset now filters to the `train` split only, `proportion_empty_prompts` forces the model to rely on the mask rather than your near-constant captions, and the dataloader transform is pickle-safe for Windows worker processes.

In [1]:
# 1. Install + clone diffusers
import subprocess, sys, os
from pathlib import Path

PROJECT_ROOT   = Path(r"D:\Study\CDC Project 1\Project")
DIFFUSERS_REPO = PROJECT_ROOT / "diffusers_repo"

if not DIFFUSERS_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/huggingface/diffusers.git",
                    str(DIFFUSERS_REPO)], check=True)
else:
    print("diffusers_repo exists, skipping clone.")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(DIFFUSERS_REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(DIFFUSERS_REPO / "examples" / "controlnet" / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install",
                "accelerate", "tensorboard", "pyyaml", "--quiet"], check=True)

diffusers_repo exists, skipping clone.


CompletedProcess(args=['d:\\Study\\CDC Project 1\\Project\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'accelerate', 'tensorboard', 'pyyaml', '--quiet'], returncode=0)

In [2]:
# 2. Blackwell (sm_120) essentials
# bitsandbytes is the single biggest VRAM win available to you. Older wheels have
# no sm_120 kernels, so a plain `pip install bitsandbytes` may import but fail at
# runtime -- we install fresh AND actually exercise the optimizer to prove it works.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "bitsandbytes"], check=False)

CompletedProcess(args=['d:\\Study\\CDC Project 1\\Project\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-U', 'bitsandbytes'], returncode=0)

In [3]:
# 3. Hardware capability check
import torch

print("torch            :", torch.__version__)
print("cuda available   :", torch.cuda.is_available())

cuda_ok = False
if torch.cuda.is_available():
    print("device           :", torch.cuda.get_device_name(0))
    print("compute capability:", torch.cuda.get_device_capability(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"total VRAM       : {total:.1f} GB")
    try:
        a = torch.randn(1024, 1024, device="cuda", dtype=torch.float16)
        _ = (a @ a).float().sum().item()
        torch.cuda.synchronize()
        cuda_ok = True
        print("fp16 matmul      : OK")
    except RuntimeError as e:
        print("fp16 matmul FAILED:", e)
        print(">>> torch build lacks sm_120 kernels. Install the cu128 build:")
        print(">>> pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu128")
else:
    print(">>> CUDA unavailable -- check driver/torch.")

xformers_ok = False
try:
    import xformers
    print("xformers         :", xformers.__version__)
    xformers_ok = True
except Exception as e:
    print("xformers         : not installed -> PyTorch SDPA will be used.")
    print("                   (SDPA is equivalent here. This is NOT a problem.)")

# Import is not enough -- run one real 8-bit step.
bnb_ok = False
try:
    import bitsandbytes as bnb
    p = torch.nn.Parameter(torch.randn(64, 64, device="cuda"))
    opt = bnb.optim.AdamW8bit([p], lr=1e-5)
    (p.sum()).backward()
    opt.step(); opt.zero_grad()
    torch.cuda.synchronize()
    bnb_ok = True
    print("bitsandbytes     :", bnb.__version__, "-> 8-bit Adam WORKS")
except Exception as e:
    print("bitsandbytes     : unusable ->", repr(e)[:160])
    print(">>> Without it you need ~2.2 GB more VRAM and will likely spill to system RAM.")
    print(">>> Try: pip install -U bitsandbytes   (needs a build with sm_120 kernels)")

if not cuda_ok:
    raise RuntimeError("Fix CUDA/torch first -- see above.")
print("\nSummary: xformers_ok =", xformers_ok, "| bnb_ok =", bnb_ok)

torch            : 2.11.0+cu128
cuda available   : True
device           : NVIDIA GeForce RTX 5060 Laptop GPU
compute capability: (12, 0)
total VRAM       : 8.5 GB
fp16 matmul      : OK
xformers         : not installed -> PyTorch SDPA will be used.
                   (SDPA is equivalent here. This is NOT a problem.)


W0806 00:32:26.786000 35676 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


bitsandbytes     : 0.50.0 -> 8-bit Adam WORKS

Summary: xformers_ok = False | bnb_ok = True


### About "Prefer No Sysmem Fallback"

Do this once, outside the notebook:

**NVIDIA Control Panel -> Manage 3D Settings -> CUDA - Sysmem Fallback Policy -> Prefer No Sysmem Fallback**

By default Windows lets CUDA silently spill VRAM into system RAM. Training keeps running but 50x slower — exactly the symptom you had. With fallback off you get an honest OOM instead, so a bad config fails in seconds rather than wasting a night.

In [4]:
# 4. Write the patched training script
#    Patches ONLY make_train_dataset(). Everything else in the official
#    train_controlnet.py (training loop, checkpointing, LR schedule) is untouched.
controlnet_examples_dir = DIFFUSERS_REPO / "examples" / "controlnet"
assert (controlnet_examples_dir / "train_controlnet.py").exists(), \
    f"Missing train_controlnet.py in {controlnet_examples_dir} -- rerun cell 1."

patched_script_path = controlnet_examples_dir / "train_controlnet_patched.py"

patched_script_source = '''
# Patched wrapper around the official diffusers train_controlnet.py.
#
# WHY: datasets\' imagefolder builder recursively scans every image under
# --train_data_dir as a candidate row and demands a metadata row per file.
# Our conditioning_images/ are referenced indirectly, so they get flagged as
# orphans -> "doesn\'t have metadata" ValueError (diffusers #7267, #6077).
# FIX: build the Dataset straight from metadata.jsonl, never touching the
# folder-scanning path.
#
# This version also:
#   * keeps only rows with split == "train" (no val leakage into training)
#   * uses module-level transform fns bound with functools.partial, so the
#     dataset pickles cleanly for Windows DataLoader worker processes
#   * skips Resize/CenterCrop when data is already square at --resolution
import json
import random
import functools
from pathlib import Path

import numpy as np
import torch
import train_controlnet as tc
from datasets import Dataset
from datasets import Image as DatasetsImage
from torchvision import transforms

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True


def _tokenize(captions, tokenizer):
    return tokenizer(captions, max_length=tokenizer.model_max_length,
                     padding="max_length", truncation=True,
                     return_tensors="pt").input_ids


def _preprocess(examples, cfg):
    captions = []
    for caption in examples[cfg["caption_column"]]:
        if random.random() < cfg["proportion_empty_prompts"]:
            captions.append("")
        elif isinstance(caption, str):
            captions.append(caption)
        elif isinstance(caption, (list, np.ndarray)):
            captions.append(random.choice(caption))
        else:
            raise ValueError("caption column must hold strings or lists of strings")

    images = [cfg["image_tf"](im.convert("RGB")) for im in examples[cfg["image_column"]]]
    conds = [cfg["cond_tf"](im.convert("RGB")) for im in examples[cfg["cond_column"]]]

    examples["pixel_values"] = images
    examples["conditioning_pixel_values"] = conds
    examples["input_ids"] = _tokenize(captions, cfg["tokenizer"])
    return examples


def make_train_dataset_patched(args, tokenizer, accelerator):
    if args.dataset_name is not None:
        dataset = tc.load_dataset(args.dataset_name, args.dataset_config_name,
                                  cache_dir=args.cache_dir, data_dir=args.train_data_dir)
    else:
        data_dir = Path(args.train_data_dir)
        metadata_path = data_dir / "metadata.jsonl"
        if not metadata_path.exists():
            raise FileNotFoundError(f"Expected {metadata_path}")

        records, n_val = [], 0
        with open(metadata_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                if row.get("split", "train") != "train":
                    n_val += 1
                    continue
                row["image"] = str(data_dir / row["image"])
                row["conditioning_image"] = str(data_dir / row["conditioning_image"])
                records.append(row)

        if not records:
            raise ValueError(f"No train-split records in {metadata_path}")
        print(f"[dataset] {len(records)} train rows ({n_val} val rows held out)")

        # "classes" is a list column of varying length -- drop it, the trainer
        # does not use it and ragged columns confuse Dataset.from_list typing.
        records = [{k: v for k, v in r.items() if k != "classes"} for r in records]

        split = Dataset.from_list(records)
        split = split.cast_column("image", DatasetsImage())
        split = split.cast_column("conditioning_image", DatasetsImage())
        dataset = {"train": split}

    column_names = dataset["train"].column_names
    image_column = args.image_column or column_names[0]
    caption_column = args.caption_column or column_names[1]
    cond_column = args.conditioning_image_column or column_names[2]
    for nm, col in (("image", image_column), ("caption", caption_column),
                    ("conditioning_image", cond_column)):
        if col not in column_names:
            raise ValueError(f"--{nm}_column {col!r} not in {column_names}")

    # Notebook 02 already letterboxes to a square at --resolution, so Resize +
    # CenterCrop would be a no-op. Keep them only as a safety net for other data.
    base = [transforms.Resize(args.resolution,
                              interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(args.resolution)]
    image_tf = transforms.Compose(base + [transforms.ToTensor(),
                                          transforms.Normalize([0.5], [0.5])])
    cond_tf = transforms.Compose(
        [transforms.Resize(args.resolution,
                           interpolation=transforms.InterpolationMode.NEAREST),
         transforms.CenterCrop(args.resolution),
         transforms.ToTensor()])

    cfg = {"image_column": image_column, "caption_column": caption_column,
           "cond_column": cond_column, "tokenizer": tokenizer,
           "proportion_empty_prompts": args.proportion_empty_prompts,
           "image_tf": image_tf, "cond_tf": cond_tf}

    with accelerator.main_process_first():
        if args.max_train_samples is not None:
            dataset["train"] = dataset["train"].shuffle(seed=args.seed).select(
                range(args.max_train_samples))
        train_dataset = dataset["train"].with_transform(
            functools.partial(_preprocess, cfg=cfg))

    return train_dataset


tc.make_train_dataset = make_train_dataset_patched

if __name__ == "__main__":
    tc.main(tc.parse_args())
'''

patched_script_path.write_text(patched_script_source, encoding="utf-8")

import ast
ast.parse(patched_script_source)          # fail loudly here, not 10 min into training
print("Wrote + syntax-checked:", patched_script_path)

Wrote + syntax-checked: D:\Study\CDC Project 1\Project\diffusers_repo\examples\controlnet\train_controlnet_patched.py


In [5]:
# 5. accelerate config (single GPU, fp16) + allocator tuning
import yaml, os
from pathlib import Path

accelerate_config = {
    "compute_environment": "LOCAL_MACHINE", "distributed_type": "NO",
    "downcast_bf16": "no", "gpu_ids": "0", "machine_rank": 0,
    "main_training_function": "main", "mixed_precision": "fp16",
    "num_machines": 1, "num_processes": 1, "rdzv_backend": "static",
    "same_network": True, "tpu_env": [], "tpu_use_cluster": False,
    "tpu_use_sudo": False, "use_cpu": False,
}
config_dir = Path.home() / ".cache" / "huggingface" / "accelerate"
config_dir.mkdir(parents=True, exist_ok=True)
config_path = config_dir / "default_config.yaml"
with open(config_path, "w") as f:
    yaml.safe_dump(accelerate_config, f, default_flow_style=False)
print("Wrote", config_path)

# expandable_segments cuts allocator fragmentation, which is what turns a
# "fits in 7.6 GB" run into an OOM on an 8 GB card.
TRAIN_ENV = dict(os.environ)
TRAIN_ENV["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
TRAIN_ENV["TOKENIZERS_PARALLELISM"] = "false"
print("PYTORCH_CUDA_ALLOC_CONF =", TRAIN_ENV["PYTORCH_CUDA_ALLOC_CONF"])

Wrote C:\Users\Ayush Dwivedi\.cache\huggingface\accelerate\default_config.yaml
PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True


In [6]:
# 6. Training config
import json
from pathlib import Path

CONTROLNET_ROOT = PROJECT_ROOT / "data" / "controlnet"
OUTPUT_DIR      = PROJECT_ROOT / "controlnet_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = CONTROLNET_ROOT / "metadata.jsonl"
assert METADATA_PATH.exists(), f"Missing {METADATA_PATH} -- run notebook 02 first."

records = [json.loads(l) for l in open(METADATA_PATH, encoding="utf-8")]
train_records = [r for r in records if r.get("split", "train") == "train"]
assert train_records, "No train-split rows in metadata.jsonl"
print(f"{len(train_records)} train rows / {len(records)} total")

# Validate on a held-out mask so the preview shows generalisation, not memorisation.
val_records = [r for r in records if r.get("split") == "val"] or train_records
VALIDATION_IMAGE  = str(CONTROLNET_ROOT / val_records[0]["conditioning_image"])
VALIDATION_PROMPT = val_records[0]["text"]
print("Validation sample:", Path(VALIDATION_IMAGE).name, "|", VALIDATION_PROMPT)

MODEL_NAME = "stable-diffusion-v1-5/stable-diffusion-v1-5"   # runwayml/... is gone

# ---- knobs ------------------------------------------------------------------
RESOLUTION            = 512     # must match SIZE in notebook 02
BATCH_SIZE            = 1
GRAD_ACCUM            = 4       # effective batch 4
DATALOADER_WORKERS    = 4       # if you hit a Windows pickling/spawn error, set 0
GRADIENT_CHECKPOINTING = True   # see the note below before turning this off
MAX_TRAIN_STEPS       = 24000
LEARNING_RATE         = 1e-5
PROPORTION_EMPTY      = 0.5     # forces reliance on the mask, not the caption
CHECKPOINT_EVERY      = 4000
KEEP_CHECKPOINTS      = 5       # -> keeps 8k, 12k, 16k, 20k, 24k
VALIDATE_EVERY        = 4000    # keep this: it is your overfitting detector
# -----------------------------------------------------------------------------

steps_per_epoch = len(train_records) // (BATCH_SIZE * GRAD_ACCUM)
print(f"\n{steps_per_epoch} optimizer steps per epoch "
      f"-> {MAX_TRAIN_STEPS} steps = {MAX_TRAIN_STEPS/max(steps_per_epoch,1):.1f} epochs")

train_script = patched_script_path
assert train_script.exists(), "Run cell 4 first."

base_cmd = [
    "accelerate", "launch", str(train_script),
    "--pretrained_model_name_or_path", MODEL_NAME,
    "--output_dir", str(OUTPUT_DIR),
    "--train_data_dir", str(CONTROLNET_ROOT),
    "--image_column", "image",
    "--conditioning_image_column", "conditioning_image",
    "--caption_column", "text",
    "--resolution", str(RESOLUTION),
    "--learning_rate", str(LEARNING_RATE),
    "--train_batch_size", str(BATCH_SIZE),
    "--gradient_accumulation_steps", str(GRAD_ACCUM),
    "--dataloader_num_workers", str(DATALOADER_WORKERS),
    "--proportion_empty_prompts", str(PROPORTION_EMPTY),
    "--set_grads_to_none",
    "--mixed_precision", "fp16",
    "--lr_scheduler", "constant_with_warmup",
    "--lr_warmup_steps", "500",
    "--checkpointing_steps", str(CHECKPOINT_EVERY),
    "--checkpoints_total_limit", str(KEEP_CHECKPOINTS),
    "--validation_image", VALIDATION_IMAGE,
    "--validation_prompt", VALIDATION_PROMPT,
    "--validation_steps", str(VALIDATE_EVERY),
    "--num_validation_images", "2",
    "--report_to", "tensorboard",
    "--seed", "42",
]
if GRADIENT_CHECKPOINTING:
    base_cmd.append("--gradient_checkpointing")
if xformers_ok:
    base_cmd.append("--enable_xformers_memory_efficient_attention")
if bnb_ok:
    base_cmd.append("--use_8bit_adam")
else:
    print("\n*** WARNING: 8-bit Adam unavailable. Expect ~2.2 GB more VRAM use. ***")
    print("*** If steps are slow, drop RESOLUTION to 384 and rerun notebook 02. ***")

bench_cmd = base_cmd + ["--max_train_steps", "30"]
full_cmd  = base_cmd + ["--max_train_steps", str(MAX_TRAIN_STEPS)]

print("\nBENCHMARK (30 steps):\n" + " ".join(bench_cmd))
print("\n\nFULL RUN:\n" + " ".join(full_cmd))

8200 train rows / 9647 total
Validation sample: img000012.png | Laparoscopic surgery showing ovary, uterus

2050 optimizer steps per epoch -> 24000 steps = 11.7 epochs

BENCHMARK (30 steps):
accelerate launch D:\Study\CDC Project 1\Project\diffusers_repo\examples\controlnet\train_controlnet_patched.py --pretrained_model_name_or_path stable-diffusion-v1-5/stable-diffusion-v1-5 --output_dir D:\Study\CDC Project 1\Project\controlnet_output --train_data_dir D:\Study\CDC Project 1\Project\data\controlnet --image_column image --conditioning_image_column conditioning_image --caption_column text --resolution 512 --learning_rate 1e-05 --train_batch_size 1 --gradient_accumulation_steps 4 --dataloader_num_workers 4 --proportion_empty_prompts 0.5 --set_grads_to_none --mixed_precision fp16 --lr_scheduler constant_with_warmup --lr_warmup_steps 500 --checkpointing_steps 4000 --checkpoints_total_limit 5 --validation_image D:\Study\CDC Project 1\Project\data\controlnet\conditioning_images\img000012.png

### On `GRADIENT_CHECKPOINTING`

Left **on** by default. ControlNet backprops through part of the frozen UNet, so
activations are large; at 512 on 8 GB, turning it off usually OOMs.

It costs roughly 35% speed, so it is worth *one* experiment: set it to `False`,
rerun cells 6 and 7, and watch the benchmark. If it survives 30 steps without
OOM and without dropping below ~1 step/sec, keep it off. If it OOMs or crawls,
put it back. Do this **after** you have set "Prefer No Sysmem Fallback", otherwise
a failure shows up as a slow run instead of a clean error.

In [7]:
# 7. Pre-flight: existing checkpoints + disk space
import shutil, re

ckpts = sorted(
    (int(re.search(r"checkpoint-(\d+)", p.name).group(1)), p)
    for p in OUTPUT_DIR.glob("checkpoint-*")
    if re.search(r"checkpoint-(\d+)", p.name)
)
if ckpts:
    print("Existing checkpoints:")
    for step, p in ckpts:
        gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9
        print(f"  step {step:>6}  {gb:5.2f} GB  {p.name}")
    LAST_STEP = ckpts[-1][0]
    per_ckpt = sum(f.stat().st_size for f in ckpts[-1][1].rglob("*") if f.is_file()) / 1e9
    print(f"\nLatest = step {LAST_STEP}. RESUME from here -- do not restart.")
    remaining = max(0, MAX_TRAIN_STEPS - LAST_STEP)
    print(f"Remaining to {MAX_TRAIN_STEPS}: {remaining} steps")
else:
    LAST_STEP, per_ckpt, remaining = 0, 4.0, MAX_TRAIN_STEPS
    print("No checkpoints found -- this will be a fresh run.")

free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
needed = per_ckpt * (KEEP_CHECKPOINTS + 1)      # +1: a new one is written before the oldest is pruned
print(f"\nFree disk: {free_gb:.1f} GB   |   needed: ~{needed:.1f} GB")
if free_gb < needed:
    print(">>> NOT ENOUGH DISK. Running out at 3am kills the run and you lose the night.")
    print(">>> Free space, or lower KEEP_CHECKPOINTS.")
else:
    print(">>> Disk OK.")


Existing checkpoints:
  step   8000   2.18 GB  checkpoint-8000
  step  10000   2.18 GB  checkpoint-10000
  step  12000   2.18 GB  checkpoint-12000

Latest = step 12000. RESUME from here -- do not restart.
Remaining to 24000: 12000 steps

Free disk: 334.2 GB   |   needed: ~13.1 GB
>>> Disk OK.


In [ ]:
# 7. Benchmark -- 30 steps, tells you your real s/step before you commit a night
import subprocess, time

t0 = time.time()
proc = subprocess.run(bench_cmd, env=TRAIN_ENV)
dt = time.time() - t0

if proc.returncode != 0:
    print(f"\nBenchmark FAILED (exit {proc.returncode}). Read the traceback above.")
    print("Common causes:")
    print("  * OOM                      -> set RESOLUTION 384 (rerun nb 02) or keep checkpointing on")
    print("  * pickling/spawn error     -> DATALOADER_WORKERS = 0")
    print("  * metadata/file not found  -> rerun notebook 02")
else:
    per_step = dt / 30
    print(f"\n--- {dt:.0f}s for 30 steps = {per_step:.2f} s/step ---")
    print(f"Projected {MAX_TRAIN_STEPS} steps: {per_step * MAX_TRAIN_STEPS / 3600:.1f} hours")
    if per_step > 15:
        print("\n>>> STILL SPILLING. This is not normal. Check in order:")
        print("    1. bnb_ok True in cell 3?")
        print("    2. Sysmem Fallback set to 'Prefer No Sysmem Fallback'?")
        print("    3. Any other app using VRAM? (browser, games) -- close them.")
        print("    4. Drop RESOLUTION to 384 and rerun notebook 02.")
    elif per_step > 6:
        print("\n>>> Workable but slow. Try DATALOADER_WORKERS=8, or RESOLUTION=384.")
    else:
        print("\n>>> Healthy. Launch the full run.")

### Run the full training from PowerShell, not the notebook

Jupyter holds its own CUDA context, and on an 8 GB card that alone can be the
difference between fitting and spilling. Activate your venv and paste the
`FULL RUN` command printed by cell 6. Set the allocator variable first:

```powershell
$env:PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"
```

Watch progress with `tensorboard --logdir "D:\Study\CDC Project 1\Project\controlnet_output"`.

**What to look for:** the validation images at step 2000 and 4000. They will
look rough — that is fine. What matters is whether they *follow the mask*: the
coloured regions should correspond to structures in roughly the right places.
If at 6000 steps the output ignores the mask entirely, stop and tell me — more
steps will not fix that, it means something upstream is wrong.

In [ ]:
# 9. (Fresh run only -- use cell 10 instead if you already have checkpoints)
import subprocess
subprocess.run(full_cmd, env=TRAIN_ENV, check=True)

In [ ]:
# 10. OVERNIGHT RUN -- resumes from the latest checkpoint
# --resume_from_checkpoint latest restores model, optimizer AND LR-scheduler
# state, so 12k already done + max_train_steps 24000 means it trains the
# remaining 12k. Restarting instead would throw those hours away.
resume_cmd = full_cmd + ["--resume_from_checkpoint", "latest"]

print("Paste this into PowerShell (venv activated), NOT the notebook:\n")
print('$env:PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"')
print(" ".join(f'"{a}"' if " " in a else a for a in resume_cmd))
print(f"\n~{remaining} steps remaining.")
print("Checkpoints will land at:",
      [s for s in range(CHECKPOINT_EVERY, MAX_TRAIN_STEPS + 1, CHECKPOINT_EVERY)][-KEEP_CHECKPOINTS:])


In [ ]:
# 11. (Alternative) run the overnight job from the notebook -- blocks the kernel
# Jupyter holds its own CUDA context, so PowerShell is the safer route on 8 GB.
import subprocess
subprocess.run(resume_cmd, env=TRAIN_ENV, check=True)